# Tools for a Customer Outreach Campaign _ V2

In the previous notebook (`07_03_tools_customer_outreach.ipynb`), the agent encountered a file encoding issue while reading instruction files with `FileReadTool`; in this notebook, we improve the tool by adding safer encoding handling to make file reading more reliable.


The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
import os

warnings.filterwarnings('ignore')
os.environ["OTEL_SDK_DISABLED"] = "true"

- Import libraries, APIs and LLM
- [Serper](https://serper.dev)

In [4]:
from crewai import Agent, Task, Crew

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Creating Agents

In [6]:
sales_rep_agent = Agent(
    role="نماینده فروش",
    goal="شناسایی سرنخ‌های (lead) با ارزش بالا که با "
         "پروفایل مشتری ایده‌آل ما مطابقت دارند",
    backstory=(
        "به عنوان بخشی از تیم پویای فروش در CrewAI، "
        "مأموریت شما کاوش در فضای دیجیتال برای یافتن سرنخ‌های (lead) بالقوه است. "
        "مجهز به ابزارهای پیشرفته "
        "و ذهنیت استراتژیک، شما داده‌ها، "
        "روندها و تعاملات را تحلیل می‌کنید تا "
        "فرصت‌هایی را کشف کنید که دیگران ممکن است از دست بدهند. "
        "کار شما در هموار کردن مسیر "
        "برای تعاملات معنادار و پیشبرد رشد شرکت نقش حیاتی دارد."
    ),
    allow_delegation=False,
    verbose=True
)

In [7]:
lead_sales_rep_agent = Agent(
    role="نماینده ارشد فروش",
    goal="پرورش لیدها با ارتباطات شخصی‌سازی‌شده و قانع‌کننده",
    backstory=(
        "در اکوسیستم پرجنب‌وجوش بخش فروش CrewAI، "
        "شما به عنوان پل ارتباطی بین مشتریان بالقوه "
        "و راه‌حل‌های مورد نیازشان برجسته هستید. "
        "با ایجاد پیام‌های جذاب و شخصی‌سازی‌شده، "
        "نه تنها لیدها را از محصولات ما آگاه می‌کنید "
        "بلکه باعث می‌شوید احساس دیده شدن و شنیده شدن داشته باشند. "
        "نقش شما در تبدیل علاقه به عمل، "
        "و هدایت لیدها در مسیر از کنجکاوی تا تعهد، حیاتی است."
    ),
    allow_delegation=False,
    verbose=True
)

## Creating Tools

### crewAI Tools

In [11]:
from crewai_tools import DirectoryReadTool, \
                         FileReadTool, \
                         SerperDevTool

We define a custom `UTF8FileReadTool` to extend CrewAI’s `FileReadTool` so agents can reliably read instruction files even when they use different text encodings, preventing `UnicodeDecodeError` issues.


In [12]:
from crewai_tools import FileReadTool


class UTF8FileReadTool(FileReadTool):
    def _run(
        self,
        file_path: str,
        start_line: int | None = None,
        line_count: int | None = None,
        **kwargs
    ) -> str:
        # read file with safe encodings
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                lines = f.readlines()
        except UnicodeDecodeError:
            try:
                with open(file_path, "r", encoding="utf-8-sig") as f:
                    lines = f.readlines()
            except UnicodeDecodeError:
                with open(file_path, "r", encoding="latin-1") as f:
                    lines = f.readlines()

        # apply line slicing like original tool
        if start_line is not None:
            start_line = max(start_line, 0)
        else:
            start_line = 0

        if line_count is not None:
            end_line = start_line + line_count
            lines = lines[start_line:end_line]
        else:
            lines = lines[start_line:]

        return "".join(lines)


In [14]:
directory_read_tool = DirectoryReadTool(directory='./instructions')
file_read_tool = UTF8FileReadTool()
search_tool = SerperDevTool()

In [18]:
search_tool._run(query="علیرضا اخوان پور")

{'searchParameters': {'q': 'علیرضا اخوان پور',
  'type': 'search',
  'num': 10,
  'engine': 'google'},
 'organic': [{'title': 'علیرضا اخوان پور - کلاس\u200cویژن - class.vision',
   'link': 'https://class.vision/teacher/%D8%B9%D9%84%DB%8C%D8%B1%D8%B6%D8%A7-%D8%A7%D8%AE%D9%88%D8%A7%D9%86-%D9%BE%D9%88%D8%B1/',
   'snippet': 'علیرضا اخوان پور، متخصص برجسته در حوزه هوش مصنوعی و یادگیری عمیق، با بیش از 10 سال سابقه تدریس و فعالیت حرفه\u200cای، در حال حاضر به عنوان مدیر فنی مجموعه دانش ...',
   'position': 1},
  {'title': 'علیرضا اخوان\u200cپور',
   'link': 'https://maktabkhooneh.org/teacher/alireza-akhavan-1/',
   'snippet': 'علیرضا اخوان\u200cپور، متخصص برجسته در حوزه هوش مصنوعی و یادگیری عمیق، با بیش از ۱۰ سال سابقه تدریس و فعالیت حرفه\u200cای، یکی از چهره\u200cهای شناخته\u200cشده در این حوزه است.',
   'position': 2},
  {'title': 'علیرضا اخوان پور',
   'link': 'https://www.aparat.com/cplusplus',
   'snippet': 'مبانی برنامه سازی - جلسه 14 (حلقه با for و الگوریتمهای مبتنی بر حدس و بررسی) · ع

### Custom Tool
- Create a custom tool using crewAi's [BaseTool](https://docs.crewai.com/core-concepts/Tools/#subclassing-basetool) class

In [19]:
from crewai.tools import BaseTool

- Every Tool needs to have a `name` and a `description`.
- For simplicity and classroom purposes, `SentimentAnalysisTool` will return `positive` for every text.
- When running locally, you can customize the code with your logic in the `_run` function.

In [22]:
class SentimentAnalysisTool(BaseTool):
    name: str ="Sentiment Analysis Tool"
    description: str = ("Analyzes the sentiment of text "
         "to ensure positive and engaging communication.")
    
    def _run(self, text: str) -> str:
        # Your custom code tool goes here
        return "positive"

In [24]:
sentiment_analysis_tool = SentimentAnalysisTool()

In [26]:
from crewai.tools import BaseTool
import requests
# for create an account and credit: https://chat.avalai.ir/?ref=1LMQHOW
class AvalAISearchTool(BaseTool):
    name: str = "AvalAI Search"
    description: str = "Search using AvalAI Serper API"

    def _run(self, query: str):
        api_key = os.getenv("AVALAI_API_KEY")
        response = requests.post(
            "https://api.avalai.ir/v1/search/serper-search",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json"
            },
            json={
                "query": query,
                "max_results": 5
            }
        )

        return response.json()

In [29]:
search_tool2 = AvalAISearchTool()
search_tool2._run(query="علیرضا اخوان پور")

{'results': [{'title': 'علیرضا اخوان پور - کلاس\u200cویژن - class.vision',
   'url': 'https://class.vision/teacher/%D8%B9%D9%84%DB%8C%D8%B1%D8%B6%D8%A7-%D8%A7%D8%AE%D9%88%D8%A7%D9%86-%D9%BE%D9%88%D8%B1/',
   'snippet': 'علیرضا اخوان پور، متخصص برجسته در حوزه هوش مصنوعی و یادگیری عمیق، با بیش از 10 سال سابقه تدریس و فعالیت حرفه\u200cای، در حال حاضر به عنوان مدیر فنی مجموعه دانش ...',
   'date': None,
   'last_updated': None},
  {'title': 'علیرضا اخوان\u200cپور',
   'url': 'https://maktabkhooneh.org/teacher/alireza-akhavan-1/',
   'snippet': 'علیرضا اخوان\u200cپور، متخصص برجسته در حوزه هوش مصنوعی و یادگیری عمیق، با بیش از ۱۰ سال سابقه تدریس و فعالیت حرفه\u200cای، یکی از چهره\u200cهای شناخته\u200cشده در این حوزه است.',
   'date': None,
   'last_updated': None},
  {'title': 'علیرضا اخوان\u200cپور - نیک آموز',
   'url': 'https://nikamooz.com/professors/alireza-akhavanpour/',
   'snippet': 'توسعه دهنده پایتون و فعال در پروژه های بینایی ماشین با یادگیری عمیق · مشاور و منتور هوش مصنوعی در شتاب

## Creating Tasks

- The Lead Profiling Task is using crewAI Tools.

In [28]:
lead_profiling_task = Task(
    description=(
        "راهنماها و دستورالعمل‌های موجود برای صنعت {industry} را پیدا و مطالعه کنید. "
        "سپس یک تحلیل عمیق از {lead_name}، "
        "شرکتی در حوزه {industry} "
        "که اخیراً به راه‌حل‌های ما علاقه نشان داده، انجام دهید. "
        "از تمام منابع داده‌ای موجود استفاده کنید "
        "تا یک پروفایل جامع تهیه شود، "
        "با تمرکز بر تصمیم‌گیرندگان کلیدی، "
        "تحولات اخیر کسب‌وکار، و نیازهای بالقوه‌ای "
        "که با پیشنهادات ما همسو هستند. "
        "این وظیفه برای تنظیم مؤثر استراتژی تعامل ما بسیار حیاتی است.\n"
        "هیچ چیزی را حدس نزنید و "
        "تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید."
    ),
    expected_output=(
        "یک گزارش جامع درباره {lead_name}، "
        "شامل پیشینه شرکت، "
        "افراد کلیدی، دستاوردهای اخیر، و نیازهای شناسایی‌شده. "
        "همچنین دستورالعمل مرتبط با صنعت {industry} را که مطالعه شده "
        "به‌عنوان مرجع استراتژی تعامل ذکر کنید. "
        "حوزه‌هایی را که راه‌حل‌های ما می‌توانند ارزش‌آفرینی کنند برجسته کنید "
        "و استراتژی‌های تعامل شخصی‌سازی‌شده پیشنهاد دهید."
    ),
    tools=[directory_read_tool, file_read_tool, search_tool],
    agent=sales_rep_agent,
)

- The Personalized Outreach Task is using your custom Tool `SentimentAnalysisTool`, as well as crewAI's `SerperDevTool` (search_tool).

In [31]:
personalized_outreach_task = Task(
    description=(
        "با استفاده از اطلاعات و دستورالعمل‌های به‌دست‌آمده از "
        "گزارش پروفایل سرنخ (lead) برای {lead_name}، "
        "یک کمپین بازاریابی هدفمند "
        "با هدف {key_decision_maker}، "
        "{position} شرکت {lead_name} طراحی کنید. "
        "از تمپلت و راهنمای مرتبط با صنعت {industry} که در گزارش ذکر شده "
        "به‌عنوان چارچوب اصلی نگارش استفاده کنید. "
        "این کمپین باید به {milestone} اخیر آن‌ها بپردازد "
        "و نشان دهد راه‌حل‌های ما چگونه از اهدافشان حمایت می‌کند. "
        "ارتباط شما باید با فرهنگ و ارزش‌های شرکت {lead_name} همسو باشد "
        "و درک عمیقی از کسب‌وکار و نیازهای آن‌ها نشان دهد.\n"
        "هیچ چیزی را حدس نزنید و "
        "تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید."
    ),
    expected_output=(
        "مجموعه‌ای از پیش‌نویس‌های ایمیل شخصی‌سازی‌شده "
        "برای {lead_name}، "
        "با تمرکز ویژه بر {key_decision_maker}. "
        "هر پیش‌نویس باید بر اساس تمپلت صنعت {industry} نوشته شده باشد "
        "و شامل پیامی متقاعدکننده باشد که راه‌حل‌های ما را "
        "به دستاوردهای اخیر و اهداف آینده آن‌ها پیوند دهد. "
        "لحن نوشتار باید جذاب، حرفه‌ای "
        "و همسو با هویت سازمانی {lead_name} باشد."
    ),
    tools=[directory_read_tool, file_read_tool, sentiment_analysis_tool, search_tool],
    agent=lead_sales_rep_agent,
)

## Creating the Crew

In [33]:
crew = Crew(
    agents=[sales_rep_agent, 
            lead_sales_rep_agent],
    
    tasks=[lead_profiling_task, 
           personalized_outreach_task],
	
    verbose=True,
	memory=True
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [49]:
inputs = {
    "lead_name": "کلاس وِیژن",
    "industry": "پلتفرم آموزشی آنلاین",
    "key_decision_maker": "علیرضا اخوان پور",
    "position": "CEO",
    "milestone": "product launch"
}

result = crew.kickoff(inputs=inputs)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.2                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: adaaa6d8-44fd-4a36-b7f0-d48be9558cc8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: راهنماها و دستورالعمل‌های موجود برای صنعت پلتفرم آموزشی آنلاین را پیدا و مطالعه کنید. سپس یک تحلیل عمیق   │
│  از کلاس وِیژن، شرکتی در حوزه پلتفرم آموزشی آنلاین که اخیراً به راه‌حل‌های ما علاقه نشان داده، انجام دهید. از تمام  │
│  منابع داده‌ای موجود استفاده کنید تا یک پروفایل جامع تهیه شود، با تمرکز بر تصمیم‌گیرندگان کلیدی، تحولات اخیر      │
│  کسب‌وکار، و نیازهای بالقوه‌ای که با پیشنهادات ما همسو هستند. این وظیفه برای تنظیم مؤثر استراتژی تعامل ما بسیار   │
│  حیاتی است.                                                                                                     │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│  ID: a11478f3-6e4e-468f-8432-3104698aa097                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: نماینده فروش                                                                                            │
│                                                                                                                 │
│  Task: راهنماها و دستورالعمل‌های موجود برای صنعت پلتفرم آموزشی آنلاین را پیدا و مطالعه کنید. سپس یک تحلیل عمیق   │
│  از کلاس وِیژن، شرکتی در حوزه پلتفرم آموزشی آنلاین که اخیراً به راه‌حل‌های ما علاقه نشان داده، انجام دهید. از تمام  │
│  منابع داده‌ای موجود استفاده کنید تا یک پروفایل جامع تهیه شود، با تمرکز بر تصمیم‌گیرندگان کلیدی، تحولات اخیر      │
│  کسب‌وکار، و نیازهای بالقوه‌ای که با پیشنهادات ما همسو هستند. این وظیفه برای تنظیم مؤثر استراتژی تعامل ما بسیار   │
│  حیاتی است.                                                                                                     │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Args: {'queries': ['راهنمای صنعت پلتفرم آموزشی آنلاین', 'کلاس ویژن تصمیم\u200cگیرندگان کلیدی', 'کلاس ویژن      │
│  تحولات اخیر کسب\u200cوکار', 'کلاس ویژن نیازهای بالقوه']}                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...


Tool sentiment_analysis_tool executed with result: positive...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: نماینده ارشد فروش                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ---                                                                                                            │
│                                                                                                                 │
│  **پیش‌نویس ایمیل اول: معرفی و آغاز همکاری**                                                                     │
│                                                                                                                 │
│  موضوع: همراهی CrewAI با کلاس وِیژن برای ارتقاء تجربه آموزش آنلاین                                               │
│                                                                                                                 │
│  سلام جناب آقای علیرضا اخوان پور عزیز،                                                                          │
│                                                                                                                 │
│  با احترام و ارادت، به تازگی از رونمایی محصول جدید کلاس وِیژن مطلع شدم و بسیار خوشحالیم که در مسیر نوآوری‌های     │
│  آموزشی شما قرار گرفتیم. هدف ما در CrewAI، همسویی کامل با ارزش‌ها و اهداف پلتفرم شماست و باور داریم فناوری‌های    │
│  پیشرفته هوش مصنوعی و تحلیل داده‌کاوی ما می‌توانند فرآیندهای آموزشی و بازاریابی شما را به سطحی نوین ارتقاء دهند.  │
│                                                                                                                 │
│  مایلیم فرصتی داشته باشیم تا به صورت مختصر، راهکارهایی را معرفی کنیم که به بهبود تجربه کاربری، شخصی‌سازی         │
│  یادگیری و توسعه محتوای تخصصی کمک می‌کند.                                                                        │
│                                                                                                                 │
│  آیا فرصتی برای یک جلسه کوتاه در هفته جاری یا آینده برای گفتگو فراهم است؟                                       │
│                                                                                                                 │
│  با احترام فراوان،                                                                                              │
│  [نام شما]                                                                                                      │
│  نماینده ارشد فروش CrewAI                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **پیش‌نویس ایمیل دوم: ارائه نسخه نمایشی اختصاصی**                                                               │
│                                                                                                                 │
│  موضوع: نسخه نمایشی انحصاری CrewAI برای موفقیت محصول جدید کلاس وِیژن                                             │
│                                                                                                                 │
│  سلام جناب آقای اخوان پور،                                                                                      │
│                                                                                                                 │
│  تشکر بابت توجه شما به همکاری با CrewAI. با

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: با استفاده از اطلاعات و دستورالعمل‌های به‌دست‌آمده از گزارش پروفایل سرنخ (lead) برای کلاس وِیژن، یک کمپین    │
│  بازاریابی هدفمند با هدف علیرضا اخوان پور، CEO شرکت کلاس وِیژن طراحی کنید. از تمپلت و راهنمای مرتبط با صنعت      │
│  پلتفرم آموزشی آنلاین که در گزارش ذکر شده به‌عنوان چارچوب اصلی نگارش استفاده کنید. این کمپین باید به product     │
│  launch اخیر آن‌ها بپردازد و نشان دهد راه‌حل‌های ما چگونه از اهدافشان حمایت می‌کند. ارتباط شما باید با فرهنگ و      │
│  ارزش‌های شرکت کلاس وِیژن همسو باشد و درک عمیقی از کسب‌وکار و نیازهای آن‌ها نشان دهد.                               │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│  Agent: نماینده ارشد فروش                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: adaaa6d8-44fd-4a36-b7f0-d48be9558cc8                                                                       │
│  Final Output: ---                                                                                              │
│                                                                                                                 │
│  **پیش‌نویس ایمیل اول: معرفی و آغاز همکاری**                                                                     │
│                                                                                                                 │
│  موضوع: همراهی CrewAI با کلاس وِیژن برای ارتقاء تجربه آموزش آنلاین                                               │
│                                                                                                                 │
│  سلام جناب آقای علیرضا اخوان پور عزیز،                                                                          │
│                                                                                                                 │
│  با احترام و ارادت، به تازگی از رونمایی محصول جدید کلاس وِیژن مطلع شدم و بسیار خوشحالیم که در مسیر نوآوری‌های     │
│  آموزشی شما قرار گرفتیم. هدف ما در CrewAI، همسویی کامل با ارزش‌ها و اهداف پلتفرم شماست و باور داریم فناوری‌های    │
│  پیشرفته هوش مصنوعی و تحلیل داده‌کاوی ما می‌توانند فرآیندهای آموزشی و بازاریابی شما را به سطحی نوین ارتقاء دهند.  │
│                                                                                                                 │
│  مایلیم فرصتی داشته باشیم تا به صورت مختصر، راهکارهایی را معرفی کنیم که به بهبود تجربه کاربری، شخصی‌سازی         │
│  یادگیری و توسعه محتوای تخصصی کمک می‌کند.                                                                        │
│                                                                                                                 │
│  آیا فرصتی برای یک جلسه کوتاه در هفته جاری یا آینده برای گفتگو فراهم است؟                                       │
│                                                                                                                 │
│  با احترام فراوان،                                                                                              │
│  [نام شما]                                                                                                      │
│  نماینده ارشد فروش CrewAI                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **پیش‌نویس ایمیل دوم: ارائه نسخه نمایشی اختصاصی**                                                               │
│                                                                                                                 │
│  موضوع: نسخه نمایشی انحصاری CrewAI برای موفقیت محصول جدید کلاس وِیژن                                             │
│                                                                                                                 │
│  سلام جناب آقای اخوان پور،                                                                                      │
│                                                                                                                 │
│  تشکر بابت توجه شما به همکاری با CrewAI. ب

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [53]:
from IPython.display import HTML
import markdown

html_content = markdown.markdown(result.raw)

HTML(f'''
<style>
    .rtl-content * {{
        direction: rtl !important;
        text-align: right !important;
    }}
</style>
<div class="rtl-content" style="
    direction: rtl;
    text-align: right; 
    font-family: Tahoma, Arial, sans-serif; 
    font-size: 15px; 
    line-height: 2;
    padding: 20px;
">
    {html_content}
</div>
''')


Reset the crew’s memories to ensure previous context does not influence the next run and to allow the agents to use the tools again from a clean state.

In [35]:
crew.reset_memories('all')


[2026-05-11 10:20:41][INFO]: [Crew (crew)] Memory memory has been reset

[2026-05-11 10:20:41][INFO]: [Crew (crew)] Task Output memory has been reset


In [37]:
inputs = {
    "lead_name": "کلاس وِیژن",
    "industry": "پلتفرم آموزشی آنلاین",
    "key_decision_maker": "علیرضا اخوان پور",
    "position": "CEO",
    "milestone": "product launch"
}
result = crew.kickoff(inputs=inputs)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.2                                                                                        │
│  Latest version:  1.14.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 23a63acc-8ad4-4fea-bcef-888956ef7718                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: راهنماها و دستورالعمل‌های موجود برای صنعت پلتفرم آموزشی آنلاین را پیدا و مطالعه کنید. سپس یک تحلیل عمیق   │
│  از کلاس وِیژن، شرکتی در حوزه پلتفرم آموزشی آنلاین که اخیراً به راه‌حل‌های ما علاقه نشان داده، انجام دهید. از تمام  │
│  منابع داده‌ای موجود استفاده کنید تا یک پروفایل جامع تهیه شود، با تمرکز بر تصمیم‌گیرندگان کلیدی، تحولات اخیر      │
│  کسب‌وکار، و نیازهای بالقوه‌ای که با پیشنهادات ما همسو هستند. این وظیفه برای تنظیم مؤثر استراتژی تعامل ما بسیار   │
│  حیاتی است.                                                                                                     │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│  ID: a3226d8b-f269-4f0a-baee-87afbed96c37                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: نماینده فروش                                                                                            │
│                                                                                                                 │
│  Task: راهنماها و دستورالعمل‌های موجود برای صنعت پلتفرم آموزشی آنلاین را پیدا و مطالعه کنید. سپس یک تحلیل عمیق   │
│  از کلاس وِیژن، شرکتی در حوزه پلتفرم آموزشی آنلاین که اخیراً به راه‌حل‌های ما علاقه نشان داده، انجام دهید. از تمام  │
│  منابع داده‌ای موجود استفاده کنید تا یک پروفایل جامع تهیه شود، با تمرکز بر تصمیم‌گیرندگان کلیدی، تحولات اخیر      │
│  کسب‌وکار، و نیازهای بالقوه‌ای که با پیشنهادات ما همسو هستند. این وظیفه برای تنظیم مؤثر استراتژی تعامل ما بسیار   │
│  حیاتی است.                                                                                                     │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: list_files_in_directory                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#2) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_memory                                                                                            │
│  Output: Found memories:                                                                                        │
│  - (score=0.86) کلاس ویژن نیاز به راهکارهایی برای گسترش آموزش آنلاین با کیفیت و افزایش تعاملات زنده و ضبط شده   │
│  دارد.                                                                                                          │
│    categories: online education, quality improvement                                                            │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['online education', 'interactive learning', 'quality enhancement']                                  │
│  - (score=0.85) کلاس ویژن یک پلتفرم آموزشی آنلاین است که در زمینه ارائه دوره‌های تخصصی هوش مصنوعی، یادگیری عمیق  │
│  و بینایی کامپیوتر فعالیت می‌کند.                                                                                │
│    categories: education, technology, artificial intelligence                                                   │
│    entities: ['کلاس ویژن']                                                                                      │
│    dates: []                                                                                                    │
│    topics: ['آموزش آنلاین', 'هوش مصنوعی', 'یادگیری عمیق', 'بینایی کامپیوتر']                                    │
│  - (score=0.84) کلاس ویژن اخیراً مجوزهای جدید برای انتشار دوره‌های آموزشی گرفته و جلسات آموزشی تخصصی مرتبط با     │
│  هوش مصنوعی را برگزار کرده است.                                                                                 │
│    categories: آموزش, هوش مصنوعی                                                                                │
│    entities: ['کلاس ویژن']                                                                                      │
│    dates: []                                                                                                    │
│    topics: ['دوره\u200cهای آموزشی', 'هوش مصنوعی']                                                               │
│  - (score=0.83) کلاس ویژن نیاز به مدیریت و ارائه محتوای آموزشی سازمانی برای شرکت‌ها و سازمان‌ها دارد.             │
│    categories: آموزش, مدیریت                                                                                    │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['مدیریت محتوا', 'آموزش سازمانی']                                                                    │
│  - (score=0.83) را‌ه‌حل‌های ما می‌توانند به ارائه راه‌حل‌های پیشرفته برای افزایش کیفیت آموزش آنلاین و تعامل           │
│  دانشجویان کمک کنند.                                                                                            │
│    categories: education, online learning, technology solutions                                                 │
│    entities: []                                                                                                 │
│    dates: []                                                                                                    │
│    topics: ['education quality', 'student engagement', 'online solutions']                                      │
│  - (score=0.83) کلاس ویژن نیاز به ارتقاء تج

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: نماینده ارشد فروش                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  پیش‌نویس‌های ایمیل شخصی‌سازی‌شده برای علیرضا اخوان پور، CEO شرکت کلاس ویژن، متناسب با کمپین بازاریابی و محصول      │
│  جدید کلاس ویژن:                                                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **موضوع:** حمایت از پیشرفت‌های کلاس ویژن در آموزش هوش مصنوعی با راهکارهای نوین ما                               │
│                                                                                                                 │
│  سلام آقای اخوان پور عزیز،                                                                                      │
│                                                                                                                 │
│  با احترام و تحسین تلاش‌های ارزشمند شما در توسعه آموزش‌های تخصصی هوش مصنوعی در کلاس ویژن، خوشحالیم که شاهد        │
│  مجوزهای جدید و برگزاری جلسات تخصصی هوش مصنوعی توسط تیم شما هستیم. این دستاوردها نشان‌دهنده تعهد شما به ارتقاء   │
│  کیفیت آموزش و توسعه دانش در این حوزه حیاتی است.                                                                │
│                                                                                                                 │
│  ما در CrewAI مفتخریم که می‌توانیم با ارائه راهکارهای یکپارچه نرم‌افزاری و فناوری‌های نوین، به گسترش آموزش آنلاین  │
│  با کیفیت و افزایش تعاملات زنده و ضبط شده شما کمک کنیم. سیستم ما تجربه آموزشی را بهبود می‌بخشد، مدیریت محتوا و   │
│  کلاس‌ها را به ساده‌ترین شکل ممکن انجام می‌دهد و موجب افزایش بهره‌وری در آموزش سازمانی شما می‌گردد.                  │
│                                                                                                                 │
│  مایلیم فرصتی برای معرفی دقیق‌تر قابلیت‌های ما داشته باشیم تا بتوانیم همکاری بلندمدتی شکل دهیم که اهداف رشد کلاس  │
│  ویژن را در زمینه آموزش هوش مصنوعی و یادگیری عمیق به بهترین شکل حمایت نماید.                                    │
│                                                                                                                 │
│  منتظر نظر شما برای هماهنگی جلسه‌ای کوتاه هستیم.                                                                 │
│                                                                                                                 │
│  با احترام،                                                                                                     │
│  [نام شما]                                                                                                      │
│  نماینده ارشد فروش CrewAI                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **موضوع:** چگونه تیم شما می‌تواند آموزش‌های سازمانی کلاس ویژن را به سطح بعدی برساند                              │
│                                  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: با استفاده از اطلاعات و دستورالعمل‌های به‌دست‌آمده از گزارش پروفایل سرنخ (lead) برای کلاس وِیژن، یک کمپین    │
│  بازاریابی هدفمند با هدف علیرضا اخوان پور، CEO شرکت کلاس وِیژن طراحی کنید. از تمپلت و راهنمای مرتبط با صنعت      │
│  پلتفرم آموزشی آنلاین که در گزارش ذکر شده به‌عنوان چارچوب اصلی نگارش استفاده کنید. این کمپین باید به product     │
│  launch اخیر آن‌ها بپردازد و نشان دهد راه‌حل‌های ما چگونه از اهدافشان حمایت می‌کند. ارتباط شما باید با فرهنگ و      │
│  ارزش‌های شرکت کلاس وِیژن همسو باشد و درک عمیقی از کسب‌وکار و نیازهای آن‌ها نشان دهد.                               │
│  هیچ چیزی را حدس نزنید و تنها از اطلاعاتی استفاده کنید که کاملاً از صحت آن‌ها مطمئن هستید.                        │
│  Agent: نماینده ارشد فروش                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 23a63acc-8ad4-4fea-bcef-888956ef7718                                                                       │
│  Final Output: پیش‌نویس‌های ایمیل شخصی‌سازی‌شده برای علیرضا اخوان پور، CEO شرکت کلاس ویژن، متناسب با کمپین          │
│  بازاریابی و محصول جدید کلاس ویژن:                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **موضوع:** حمایت از پیشرفت‌های کلاس ویژن در آموزش هوش مصنوعی با راهکارهای نوین ما                               │
│                                                                                                                 │
│  سلام آقای اخوان پور عزیز،                                                                                      │
│                                                                                                                 │
│  با احترام و تحسین تلاش‌های ارزشمند شما در توسعه آموزش‌های تخصصی هوش مصنوعی در کلاس ویژن، خوشحالیم که شاهد        │
│  مجوزهای جدید و برگزاری جلسات تخصصی هوش مصنوعی توسط تیم شما هستیم. این دستاوردها نشان‌دهنده تعهد شما به ارتقاء   │
│  کیفیت آموزش و توسعه دانش در این حوزه حیاتی است.                                                                │
│                                                                                                                 │
│  ما در CrewAI مفتخریم که می‌توانیم با ارائه راهکارهای یکپارچه نرم‌افزاری و فناوری‌های نوین، به گسترش آموزش آنلاین  │
│  با کیفیت و افزایش تعاملات زنده و ضبط شده شما کمک کنیم. سیستم ما تجربه آموزشی را بهبود می‌بخشد، مدیریت محتوا و   │
│  کلاس‌ها را به ساده‌ترین شکل ممکن انجام می‌دهد و موجب افزایش بهره‌وری در آموزش سازمانی شما می‌گردد.                  │
│                                                                                                                 │
│  مایلیم فرصتی برای معرفی دقیق‌تر قابلیت‌های ما داشته باشیم تا بتوانیم همکاری بلندمدتی شکل دهیم که اهداف رشد کلاس  │
│  ویژن را در زمینه آموزش هوش مصنوعی و یادگیری عمیق به بهترین شکل حمایت نماید.                                    │
│                                                                                                                 │
│  منتظر نظر شما برای هماهنگی جلسه‌ای کوتاه هستیم.                                                                 │
│                                                                                                                 │
│  با احترام،                                                                                                     │
│  [نام شما]                                                                                                      │
│  نماینده ارشد فروش CrewAI                                                                                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **موضوع:** چگونه تیم شما می‌تواند آموزش‌های سازمانی کلاس ویژن را به سطح بعدی برساند                              │
│                                 

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [39]:
from IPython.display import HTML
import markdown

html_content = markdown.markdown(result.raw)

HTML(f'''
<style>
    .rtl-content * {{
        direction: rtl !important;
        text-align: right !important;
    }}
</style>
<div class="rtl-content" style="
    direction: rtl;
    text-align: right; 
    font-family: Tahoma, Arial, sans-serif; 
    font-size: 15px; 
    line-height: 2;
    padding: 20px;
">
    {html_content}
</div>
''')
